## Lesson Overview

**What this lesson teaches:** how to stop eyeballing individual model outputs and instead build a repeatable, scored benchmark for a prompt. The trick is using Claude for two jobs it isn't usually asked to do at once — inventing the test cases *and* grading the results — so you can compare prompt versions against a consistent number instead of a gut feeling.

**What's happening under the hood, step by step:**
1. **Setup cell** loads `.env` and creates one shared `client`/`MODEL_NAME` that every later cell reuses.
2. **`generate_unique_ideas`** makes one API call asking Claude for N distinct test scenarios. It forces clean JSON back by pre-filling the assistant turn with `` ```json `` and cutting the response off at the next `` ``` `` (`stop_sequences=["```"]`) — no prose to strip out afterward.
3. **`generate_test_case`** turns each idea into a full test case (concrete inputs + grading criteria) with the same JSON-forcing trick. `generate_dataset` fires this once per idea **concurrently** via a `ThreadPoolExecutor` (`max_concurrent_tasks` caps how many run at once — raise for speed, lower if you hit rate limits), then writes everything to `dataset.json`.
4. **`run_prompt`** is the prompt you're actually testing — it's not part of the framework, it's the thing being measured.
5. **`run_evaluation`** loads `dataset.json` and, again concurrently, calls `run_prompt` on each test case's inputs and then `grade_output` — a separate API call where Claude scores that specific output 1-10 against *that test case's own* criteria (with a hard rule: violating a mandatory requirement caps the score at 3).
6. Results are averaged, then written to `output.json` and rendered into a color-coded `output.html` report by `generate_prompt_evaluation_report`.

In short: Claude designs the exam, you take it, and Claude grades it — all orchestrated by plain Python doing JSON parsing, thread pools, and file I/O in between the API calls.


# Lesson 6: Building an automated prompt evaluation framework

This notebook builds a small framework for testing a prompt against many scenarios at once, instead of eyeballing one output at a time:
- it generates a synthetic dataset of test cases from a task description
- it runs your prompt against every test case
- it asks Claude to grade each output against that test case's own criteria
- it summarizes the results as JSON and as a browsable HTML report

Unlike Lessons 7 and 8, this lesson isn't about tool-calling — it's about using the model itself as a *test case generator* and a *grader*, so you can iterate on a prompt against a repeatable, scored benchmark instead of manual spot checks.

## How This Notebook Works

The lesson follows a pipeline:
1. load environment variables and build the Anthropic client
2. define small message helpers (`add_user_message`, `add_assistant_message`, `chat`) reused throughout
3. define `generate_prompt_evaluation_report`, which turns raw results into an HTML report
4. define `PromptEvaluator`, the class that generates test cases, runs your prompt, and grades the output
5. create a `PromptEvaluator` instance and generate a dataset of test cases
6. define the prompt you actually want to evaluate
7. run the evaluation and inspect the JSON/HTML output

Each step builds on the one before it, so run the cells in order the first time through.

## Setup

Install the dependencies once in your environment:

```bash
pip install anthropic python-dotenv
```

Put your credentials in `Claude_API_Training/.env` or the project root `.env` file. A minimal file looks like this:

```env
ANTHROPIC_API_KEY=your_key_here
MODEL_NAME=claude-haiku-4-5
```

`MODEL_NAME` is optional. If it is missing, the notebook falls back to `claude-haiku-4-5`.

This lesson calls the API repeatedly (once per generated idea, once per test case, once per grade), so make sure your key is valid before running the dataset-generation and evaluation cells — those steps do nothing useful without it.

In [ ]:
import concurrent.futures
import json
import os
import re
from pathlib import Path
from statistics import mean
from textwrap import dedent

try:
    import anthropic
except ImportError:
    anthropic = None

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

def load_env_file(env_path):
    env_path = Path(env_path)
    if not env_path.exists():
        return False

    with env_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")
    return True

load_dotenv()

env_loaded = False
for candidate in (Path(".env"), Path("Claude_API_Training/.env")):
    if load_env_file(candidate):
        env_loaded = True
        break

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
MODEL_NAME = os.environ.get("MODEL_NAME", "claude-haiku-4-5")
client = anthropic.Anthropic(api_key=API_KEY) if anthropic and API_KEY else None

print(f"Environment loaded: {env_loaded}")
print(f"Model: {MODEL_NAME}")
if client is None:
    print("Anthropic client is not available yet. Install anthropic and set ANTHROPIC_API_KEY in .env before running the dataset or evaluation cells.")
else:
    print("Anthropic client ready.")


### Setup Notes

The setup cell is the only place environment state is loaded. After it runs, every other cell can reuse the same `client` and `MODEL_NAME`.

If `anthropic` is not installed or the API key is missing, `client` is `None`. The helper and class definitions below still load fine, but any cell that actually calls the API (dataset generation, running the evaluator) will raise a clear error until credentials are in place.

## Message helpers

Anthropic conversations are tracked as `role`/`content` dictionaries. `add_user_message` and `add_assistant_message` keep that shape consistent, and `chat` wraps `client.messages.create` so every call in this notebook goes through the same place.

This lesson never uses tools, so `chat` returns `message.content[0].text` directly — plain text, not the raw message object — which keeps the rest of the code (JSON parsing, grading, report building) simple.

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    if client is None:
        raise RuntimeError("Set up the Anthropic package and ANTHROPIC_API_KEY before calling chat().")

    params = {
        "model": MODEL_NAME,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

### Why These Helpers Matter

Every method in `PromptEvaluator` below builds a prompt, sends it through `chat`, and parses the JSON that comes back. Because `add_user_message`, `add_assistant_message`, and `chat` are the only things touching the API, a formatting bug only needs to be fixed in one place.

## HTML report builder

`generate_prompt_evaluation_report` takes the list of graded results and renders them as a self-contained HTML page: summary stats at the top (total tests, average score, pass rate), then one row per test case with its inputs, criteria, output, score, and the grader's reasoning.

You don't need to read the CSS closely — the part worth understanding is the loop near the bottom that builds one `<tr>` per result and color-codes the score cell (green ≥8, yellow 6-7, red ≤5).

## The PromptEvaluator class

This class is the core of the lesson. Each method hands off to the next:

- **`render`** — a tiny template engine that replaces `{placeholder}` tokens in a prompt string with real values, and unescapes `{{`/`}}` back to literal braces (needed because the prompts below contain literal JSON braces).
- **`generate_unique_ideas`** — asks Claude for N distinct testing scenarios for your task (e.g. "a low-risk, long-horizon investor" vs. "a high-risk, short-horizon investor"). Returns a plain JSON array of short descriptions.
- **`generate_test_case`** — turns one scenario into a full test case: concrete `prompt_inputs` plus a short list of `solution_criteria` the output will be graded against.
- **`generate_dataset`** — calls the two methods above for every idea (in a thread pool, so cases generate concurrently) and saves the full list to `dataset.json`.
- **`grade_output`** — the grader. Given a test case and an actual output, asks Claude to score it 1-10 against that test case's own `solution_criteria`, with strict rules about mandatory-requirement violations capping the score at 3.
- **`run_test_case`** — runs your prompt function on one test case's inputs, then grades the result.
- **`run_evaluation`** — loads `dataset.json`, runs and grades every test case concurrently, prints the average score, and writes both `output.json` and `output.html` (via the report builder above).

The prompts inside `generate_unique_ideas` and `generate_test_case` both end the assistant turn with `` ```json `` and pass `stop_sequences=["```"]` — a common pattern for forcing clean, parseable JSON output without any surrounding prose.

In [ ]:
def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin-top: 5px;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
            .output-cell {{
                white-space: pre-wrap;
            }}
            .score {{
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                display: inline-block;
            }}
            .score-high {{
                background-color: #c8e6c9;
                color: #2e7d32;
            }}
            .score-medium {{
                background-color: #fff9c4;
                color: #f57f17;
            }}
            .score-low {{
                background-color: #ffcdd2;
                color: #c62828;
            }}
            .output {{
                overflow: auto;
                white-space: pre-wrap;
            }}

            .output pre {{
                background-color: #f5f5f5;
                border: 1px solid #ddd;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 14px;
                line-height: 1.4;
                color: #333;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap; 
                word-wrap: break-word; 
            }}

            td {{
                width: 20%;
            }}
            .score-col {{
                width: 80px;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total_tests}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg_score:.1f} / {max_possible_score}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥7)</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt Inputs</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [
                f"<strong>{key}:</strong> {value}"
                for key, value in result["test_case"]["prompt_inputs"].items()
            ]
        )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        score = result["score"]
        if score >= 8:
            score_class = "score-high"
        elif score <= 5:
            score_class = "score-low"
        else:
            score_class = "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td class="prompt-inputs">{prompt_inputs_html}</td>
                <td class="criteria">• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """

    return html

In [ ]:
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}]+)}", template_string)

        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace(
                    "{" + placeholder + "}", str(variables[placeholder])
                )

        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        """Generate a list of unique ideas for test cases based on the task description"""

        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:
        
        <task_description>
        {task_description}
        </task_description>

        The prompt will receive the following inputs
        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>
        
        Each idea should represent a distinct scenario or example that tests different aspects of the task.
        
        Output Format:
        Provide your response as a structured JSON array where each item is a brief description of the idea.
        
        Example:
        ```json
        [
            "Testing with technical computer science terminology",
            "Testing with medical research findings",
            "Testing with complex mathematical concepts",
            ...
        ]
        ```
        
        Ensure each idea is:
        - Clearly distinct from the others
        - Relevant to the task description
        - Specific enough to guide generation of a full test case
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output

        Remember, only generate {num_cases} unique ideas
        """

        system_prompt = "You are a test scenario designer specialized in creating diverse, unique testing scenarios."

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=1.0,
        )

        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        """Generate a single test case based on the task description and a specific idea"""

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec.keys()])

        prompt = """
        Generate a single detailed test case for a prompt evaluation based on:
        
        <task_description>
        {task_description}
        </task_description>
        
        <specific_idea>
        {idea}
        </specific_idea>
        
        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>
        
        Output Format:
        ```json
        {{
            "prompt_inputs": {{
            {example_prompt_inputs}
            }},
            "solution_criteria": ["criterion 1", "criterion 2", ...] // Concise list of criteria for evaluating the solution, 1 to 4 items
        }}
        ```
        
        IMPORTANT REQUIREMENTS:
        - You MUST ONLY use these exact input keys in your prompt_inputs: {allowed_keys}        
        - Do NOT add any additional keys to prompt_inputs
        - All keys listed in allowed_input_keys must be included in your response
        - Make the test case realistic and practically useful
        - Include measurable, concise solution criteria
        - The solution criteria should ONLY address the direct requirements of the task description and the generated prompt_inputs
        - Avoid over-specifying criteria with requirements that go beyond the core task
        - Keep solution criteria simple, focused, and directly tied to the fundamental task
        - The test case should be tailored to the specific idea provided
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output
        - DO NOT include any fields beyond those specified in the output format

        Here's an example of a sample input with an ideal output:
        <sample_input>
        <sample_task_description>
        Extract topics out of a passage of text
        </sample_task_description>
        <sample_specific_idea>
        Testing with a text that contains multiple nested topics and subtopics (e.g., a passage about renewable energy that covers solar power economics, wind turbine technology, and policy implications simultaneously)
        </sample_specific_idea>

        <sample_allowed_input_keys>
        "content"
        </sample_allowed_input_keys>
        </sample_input>
        <ideal_output>
        ```json
        {
            "prompt_inputs": {
                "content": "The transition to renewable energy encompasses numerous interdependent dimensions. Solar photovoltaic technology has seen dramatic cost reductions, with panel efficiency improving 24% since 2010 while manufacturing costs declined by 89%, making it economically competitive with fossil fuels in many markets. Concurrently, wind energy has evolved through innovative turbine designs featuring carbon-fiber composite blades and advanced control systems that increase energy capture by 35% in low-wind conditions."
            },
            "solution_criteria": [
                "Includes all topics mentioned"   
            ]
        }
        ```
        </ideal_output>
        This is ideal output because the solution criteria is concise and doesn't ask for anything outside of the scope of the task description.
        """

        system_prompt = "You are a test case creator specializing in designing evaluation scenarios."

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=0.7,
        )

        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea

        return test_case

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec={},
        num_cases=1,
        output_file="dataset.json",
    ):
        """Generate test dataset based on task description and save to file"""
        ideas = self.generate_unique_ideas(
            task_description, prompt_inputs_spec, num_cases
        )

        dataset = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_idea = {
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20

                    if milestone_percentage > last_reported_percentage:
                        print(f"Generated {completed}/{total} test cases")
                        last_reported_percentage = milestone_percentage

                    dataset.append(result)
                except Exception as e:
                    print(f"Error generating test case: {e}")

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)

        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        """Grade the output of a test case using the model"""

        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            Mandatory Requirements - ANY VIOLATION MEANS AUTOMATIC FAILURE (score of 3 or lower):
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        Your task is to evaluate the following AI-generated solution with EXTREME RIGOR.

        Original task description:
        <task_description>
        {task_description}
        </task_description>

        Original task inputs:
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Scoring Guidelines:
        * Score 1-3: Solution fails to meet one or more MANDATORY requirements
        * Score 4-6: Solution meets all mandatory requirements but has significant deficiencies in secondary criteria
        * Score 7-8: Solution meets all mandatory requirements and most secondary criteria, with minor issues
        * Score 9-10: Solution meets all mandatory and secondary criteria

        IMPORTANT SCORING INSTRUCTIONS:
        * Grade the output based ONLY on the listed criteria. Do not add your own extra requirements.
        * If a solution meets all of the mandatory and secondary criteria give it a 10
        * Don't complain that the solution "only" meets the mandatory and secondary criteria. Solutions shouldn't go above and beyond - they should meet the exact listed criteria.
        * ANY violation of a mandatory requirement MUST result in a score of 3 or lower
        * The full 1-10 scale should be utilized - don't hesitate to give low scores when warranted

        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(
            messages,
            stop_sequences=["```"],
            temperature=0.0,
        )
        return json.loads(eval_text)

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        """Run a test case and grade the result"""
        output = run_prompt_function(test_case["prompt_inputs"])

        model_grade = self.grade_output(test_case, output, extra_criteria)
        model_score = model_grade["score"]
        reasoning = model_grade["reasoning"]

        return {
            "output": output,
            "test_case": test_case,
            "score": model_score,
            "reasoning": reasoning,
        }

    def run_evaluation(
        self,
        run_prompt_function,
        dataset_file,
        extra_criteria=None,
        json_output_file="output.json",
        html_output_file="output.html",
    ):
        """Run evaluation on all test cases in the dataset"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_prompt_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in concurrent.futures.as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20

                if milestone_percentage > last_reported_percentage:
                    print(f"Graded {completed}/{total} test cases")
                    last_reported_percentage = milestone_percentage
                results.append(result)

        average_score = mean([result["score"] for result in results])
        print(f"Average score: {average_score}")

        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)

        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

## Create the evaluator

Everything above is definitions. This is the first cell that actually does something: it creates a `PromptEvaluator` instance that the rest of the notebook reuses.

`max_concurrent_tasks` controls how many API calls run in parallel during dataset generation and evaluation. Raise it for speed, but watch for rate-limit errors if you push it too high.

In [ ]:
evaluator = PromptEvaluator(max_concurrent_tasks=1)

## Generate a test dataset

`generate_dataset` is where the framework actually creates its benchmark. Give it:
- `task_description` — what the prompt you're testing is supposed to do
- `prompt_inputs_spec` — the input fields your prompt takes, with a short description of each
- `num_cases` — how many synthetic test cases to generate
- `output_file` — where to save the resulting `dataset.json`

Under the hood this calls `generate_unique_ideas` once, then `generate_test_case` once per idea (concurrently), and writes the combined list to disk. If you're hitting rate limits, lower `num_cases` or `max_concurrent_tasks` above.

In [ ]:
dataset = evaluator.generate_dataset(
    task_description="Generate one concise investment idea tailored to the user's profile, with a brief rationale and key risks.",
    prompt_inputs_spec={
        "risk_tolerance": "low | medium | high",
        "time_horizon": "short-term | medium-term | long-term",
        "capital": "amount available to invest",
        "preferences": "industries, themes, or assets the user prefers",
        "avoid": "industries or assets the user wants to avoid",},
    output_file="dataset.json",
    num_cases=3,
)

## Define the prompt under test

`run_prompt` is the thing you're actually evaluating — everything else in this notebook exists to test it. It takes one test case's `prompt_inputs` dict and must return the raw model output as a string.

Swap out the prompt text below for whatever you want to benchmark. `run_evaluation` (next cell) calls this function once per test case, then grades what it returns.

In [ ]:
def run_prompt(prompt_inputs):
    prompt = f"""
    You are a careful investment research assistant.

        User profile:
        - Risk tolerance: {prompt_inputs["risk_tolerance"]}
        - Time horizon: {prompt_inputs["time_horizon"]}
        - Capital: {prompt_inputs["capital"]}
        - Preferences: {prompt_inputs["preferences"]}
        - Avoid: {prompt_inputs["avoid"]}

        Task:
        Generate one investment idea that fits this profile.

        Requirements:
        - Give a short title for the idea
        - Explain why it fits the profile in 2-4 sentences
        - List 2-3 key risks
        - Keep it general and educational
        - Do not claim guaranteed returns
        - Do not present this as personal financial advice
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

## Run the full evaluation

This is the cell that ties everything together: it loads `dataset.json`, runs `run_prompt` against every test case, grades each output with `grade_output`, and writes `output.json` and `output.html`.

Watch the console output for the "Graded X/Y test cases" progress messages and the final average score. Open `output.html` afterward for the full per-case breakdown.

In [12]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json"
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7.666666666666667


## Summary: how to use this notebook

1. Run the setup cell first so the notebook loads `.env` and creates the Anthropic client.
2. Run the message helper, HTML report builder, and `PromptEvaluator` class cells to load all the definitions.
3. Run the evaluator-creation cell, then `generate_dataset` to build a `dataset.json` of synthetic test cases for your task.
4. Edit `run_prompt` to call the actual prompt you want to evaluate.
5. Run `run_evaluation` to score every test case and produce `output.json` and `output.html`.
6. To evaluate a different prompt, change `task_description`, `prompt_inputs_spec`, and `run_prompt`, then regenerate the dataset and re-run the evaluation.

The main idea here is a repeatable benchmark: instead of manually judging a handful of outputs, you let Claude generate diverse test cases and grade the results, so you can compare prompt changes against a consistent score.